# Option Greeks and Arithmetic-Average Asian Options

This notebook extends the pricing engine in two directions:

1. European calls and puts expose the complete standard Greek set: delta, gamma, vega, theta and rho.
2. Fixed-strike arithmetic-average Asian calls and puts are represented as derivative objects and priced under risk-neutral GBM using Monte Carlo simulation.

The European Greeks are analytical Black–Scholes sensitivities. Asian Greeks are estimated by finite differences with common random numbers, which reduces noise by reusing identical simulated shocks across bumped valuations.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd

from src.yieldcurve import YieldCurve
from src.derivatives import EuropeanCall, EuropeanPut, AsianCall, AsianPut

## Market inputs and derivative objects

In [2]:
S0 = 100.0
K = 100.0
T = 1.0
sigma = 0.20

maturities = [0.5, 1.0, 2.0, 3.0]
zero_rates = [0.03, 0.032, 0.035, 0.038]
yield_curve = YieldCurve(maturities, zero_rates)

common_inputs = dict(
    S0=S0,
    K=K,
    T=T,
    sigma=sigma,
    yield_curve=yield_curve,
)

european_call = EuropeanCall(**common_inputs)
european_put = EuropeanPut(**common_inputs)

asian_inputs = dict(
    **common_inputs,
    n_steps=52,
    n_simulations=40_000,
    seed=42,
    antithetic=True,
)
asian_call = AsianCall(**asian_inputs)
asian_put = AsianPut(**asian_inputs)

## Monte Carlo prices and uncertainty

The Asian option classes expose both `price()` and `estimate()`. The latter includes the simulation standard error and confidence interval.

In [3]:
asian_call_estimate = asian_call.estimate()
asian_put_estimate = asian_put.estimate()

price_summary = pd.DataFrame(
    [
        {
            'Option': 'European call',
            'Price': european_call.price(),
            'Standard error': None,
            '95% CI lower': None,
            '95% CI upper': None,
        },
        {
            'Option': 'European put',
            'Price': european_put.price(),
            'Standard error': None,
            '95% CI lower': None,
            '95% CI upper': None,
        },
        {
            'Option': 'Asian call',
            'Price': asian_call_estimate.price,
            'Standard error': asian_call_estimate.standard_error,
            '95% CI lower': asian_call_estimate.confidence_interval[0],
            '95% CI upper': asian_call_estimate.confidence_interval[1],
        },
        {
            'Option': 'Asian put',
            'Price': asian_put_estimate.price,
            'Standard error': asian_put_estimate.standard_error,
            '95% CI lower': asian_put_estimate.confidence_interval[0],
            '95% CI upper': asian_put_estimate.confidence_interval[1],
        },
    ]
)
price_summary.round(4)

,Option,Price,Standard error,95% CI lower,95% CI upper
0,European call,9.5146,NaN,NaN,NaN
1,European put,6.3653,NaN,NaN,NaN
2,Asian call,5.3664,0.0390,5.2901,5.4427
3,Asian put,3.7861,0.0281,3.7309,3.8412


The Asian options are cheaper than the corresponding European options in this at-the-money example because averaging reduces the dispersion of the payoff-relevant underlying price.

## Complete Greek comparison

The internal convention is:

- delta: value change for a one-unit increase in spot;
- gamma: change in delta per one-unit increase in spot;
- vega: derivative with respect to decimal volatility;
- theta: annual value decay as calendar time passes;
- rho: derivative with respect to the decimal continuously compounded rate.

For easier interpretation, the table also scales vega and rho to a one-percentage-point move and theta to one day.

In [4]:
options = {
    'European call': european_call,
    'European put': european_put,
    'Asian call': asian_call,
    'Asian put': asian_put,
}

rows = []
for name, option in options.items():
    greek_values = option.greeks()
    rows.append(
        {
            'Option': name,
            'Delta': greek_values['delta'],
            'Gamma': greek_values['gamma'],
            'Vega (1 vol point)': 0.01 * greek_values['vega'],
            'Theta (1 day)': greek_values['theta'] / 365.0,
            'Rho (1 rate point)': 0.01 * greek_values['rho'],
        }
    )

greek_summary = pd.DataFrame(rows).set_index('Option')
greek_summary.round(4)

,Delta,Gamma,Vega (1 vol point),Theta (1 day),Rho (1 rate point)
Option,,,,,
European call,0.6026,0.0193,0.3857,-0.0150,0.5074
European put,-0.3974,0.0193,0.3857,-0.0065,-0.4611
Asian call,0.5659,0.0300,0.2230,-0.0082,0.2416
Asian put,-0.4185,0.0300,0.2246,-0.0040,-0.2468


Several expected relationships are visible:

- call delta is positive and put delta is negative;
- gamma and vega are positive for all four long-option positions;
- Asian vega is lower because averaging dampens sensitivity to volatility;
- call rho is positive while put rho is negative;
- theta is negative for these at-the-money long options.

The Asian gamma is somewhat larger in this example. Averaging reduces effective payoff volatility, so the option value can be more sharply curved around the at-the-money strike even while total vega is lower.

## Implementation note

The Asian Greek estimates use central finite differences. Every bumped valuation reuses the same normal shock matrix, so differences predominantly reflect the parameter change rather than unrelated Monte Carlo sampling noise. This is an application of the common-random-numbers variance-reduction technique.